<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementing convolutional layers with PyTorch

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images

In [3]:
sample_images = np.stack(load_sample_images()["images"])
# convert NumPy array to torch tensor and rescale the pixel values from 0-255 to 0-1
sample_images = torch.tensor(sample_images, dtype=torch.float32)/255

In [7]:
load_sample_images()["images"][0].shape

(427, 640, 3)

In [8]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [9]:
# PyTorch wants channel dimension before the height and width dimensions
sample_images_permuted = sample_images.permute(0,3,1,2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [10]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70,120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [12]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels = 3, out_channels= 32, kernel_size =7)
fmaps = conv_layer(cropped_images)

In [13]:
fmaps.shape

torch.Size([2, 32, 64, 114])

In [29]:
import torch.nn.functional as F

class DepthPool(torch.nn.Module):
  def __init__(self, kernel_size, stride=None, padding =0):
    super().__init__()
    self.kernel_size = kernel_size
    self.stride = stride if stride is not None else kernel_size
    self.padding = padding

  def forward(self, inputs):
    batch, channels, height, width = inputs.shape
    Z=inputs.reshape(batch, channels, height*width) # merge spatial dimensions
    Z = Z.permute (0,2,1) # switch spatial dimension with channels
    Z = F.max_pool1d(Z, kernel_size= self.kernel_size,
                     stride = self.stride,
                     padding = self.padding)
    Z = Z.permute(0,2,1) # switch spatial dimensions back
    return Z.reshape(batch, -1, height, width)


In [30]:
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [31]:
# let's take the depthwise max pooling across the 3 channels
depth_pooled_images = DepthPool(kernel_size=3, padding=0)(cropped_images)
depth_pooled_images.shape

torch.Size([2, 1, 70, 120])

In [34]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)
output.shape

torch.Size([2, 3, 1, 1])